In [4]:
%load_ext autoreload
%autoreload 2

# Openning data

In [5]:
# tests/preprocessing/datasets.ipynb
from pathlib import Path
import sys, os

# point to your project root
project_root = Path(r"/home/galencarmedeiro/git/postdoc/ragtree")
#project_root = Path(r"C:\Users\henri\Documents\git\post-doc\ragtree")
os.chdir(project_root)  # so relative paths go to data/, not tests/
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("CWD:", os.getcwd())

CWD: /home/galencarmedeiro/git/postdoc/ragtree


# DOCRED

#### BYOKG

In [ ]:
%run "scripts/build_kg_from_preprocessed.py" \
  --dataset-key docred_causal \
  --doc-types train_annotated \
  --skip 0

In [ ]:
%run "scripts/run_kg_rag_relations.py" \
  --dataset-key docred_causal \
  --backend vllm \
  --doc-type-filter dev \
  --kg-path data/kg/docred_causal__types=train_annotated_skip=0_limit=None__kg.json

In [ ]:
%run "scripts/eval_relations.py" \
  --dataset-key docred_causal \
  --method kg_rag \
  --backend vllm \
  --doc-type dev

#### CommunityRAG

In [11]:
%run "scripts/build_community_kgrag_index.py" \
  --dataset-key docred_causal \
  --kg-path data/kg/docred_causal__types=train_annotated_skip=0_limit=None__kg.json \
  --output-dir data/kg_community \
  --device cpu


[communitykgrag] Building graph + Louvain communities...
[communitykgrag] Embedding nodes on device=cpu ...


Batches: 100%|██████████| 1860/1860 [03:43<00:00,  8.33it/s]


[communitykgrag] built at: data/kg_community/docred_causal
  nodes=59493 edges=38180 sentences=38180 communities=4580
  community index: data/kg_community/docred_causal/faiss.community.index


In [18]:
%run "scripts/run_community_kgrag_relations.py" \
  --dataset-key docred_causal \
  --communitykg-root data/kg_community \
  --backend vllm \
  --doc-type-filter dev \
  --shot-type train \
  --shot-num 3 \
  --top-communities 3 \
  --top-sentences 3 \
  --max-ctx-chars 3000


[runner] dataset-key=docred_causal
[runner] method=community_kgrag
[runner] backend=vllm, model=openai/gpt-oss-20b
[runner] input=data/preprocessed/docred_causal.jsonl
[runner] output=/home/galencarmedeiro/git/postdoc/ragtree/data/processed/docred_causal.community_kgrag.vllm.jsonl
[runner] output-format=full
[runner] doc-type-filter=['dev']
[runner] skip=0, limit=None
[runner] Loaded DocRED rel_info with 96 entries.


[community_kgrag] Collecting few-shots (type=train): 106924doc [00:04, 24321.26doc/s]


[community_kgrag] few-shots: requested=3 collected=0 type=train shot_skip=0 shot_limit=None


Running community_kgrag on docred_causal: 106924doc [3:28:00,  8.57doc/s]   

[runner] Done. Processed 998 documents.
[runner] docs_after_type_filter=998, skip=0, limit=None
[runner] Skipped 105926 documents due to doc-type filter.


In [19]:
%run "scripts/eval_relations.py" \
  --dataset-key docred_causal \
  --method community_kgrag \
  --backend vllm \
  --doc-type dev

[eval] dataset-key: docred_causal
[eval] method: community_kgrag
[eval] backend: vllm
[eval] doc-type: dev
[eval] gold: data/preprocessed/docred_causal.jsonl
[eval] preds: /home/galencarmedeiro/git/postdoc/ragtree/data/processed/docred_causal.community_kgrag.vllm.jsonl
[eval] ignore-labels: ['null']
[eval] metrics-out: /home/galencarmedeiro/git/postdoc/ragtree/results/relations/docred_causal/community_kgrag.vllm.dev.json

=== Micro-level metrics ===
Precision: 0.1962
Recall:    0.0042
F1:        0.0081

=== Counts ===
TP: 51
FP: 209
FN: 12224
num_docs_seen: 998
num_docs_eval: 998
num_docs_missing_gold: 0

[eval] Done.


#### Triple KG RAG

In [9]:
%run "scripts/run_triple_kg_rag_relations.py" \
  --dataset-key docred_causal \
  --backend vllm \
  --doc-type-filter dev \
  --kg-max-hops 1 \
  --kg-max-triples 120 \
  --shot-type dev \
  --shot-num 3

[runner] dataset-key=docred_causal
[runner] method=triple_kg_rag
[runner] backend=vllm, model=openai/gpt-oss-20b
[runner] input=data/preprocessed/docred_causal.jsonl
[runner] output=/home/galencarmedeiro/git/postdoc/ragtree/data/processed/docred_causal.triple_kg_rag.vllm.jsonl
[runner] output-format=full
[runner] doc-type-filter=['dev']
[runner] skip=0, limit=None
[runner] Loaded DocRED rel_info with 96 entries.


[triple_kg_rag] Collecting few-shots (type=dev): 2doc [00:00, 4914.24doc/s]


[triple_kg_rag] few-shots: requested=3 collected=3 type=dev shot_skip=0 shot_limit=None


Running triple_kg_rag on docred_causal: 106924doc [4:51:41,  6.11doc/s]   

[runner] Done. Processed 998 documents.
[runner] docs_after_type_filter=998, skip=0, limit=None
[runner] Skipped 105926 documents due to doc-type filter.


In [10]:
%run "scripts/eval_relations.py" \
  --dataset-key docred_causal \
  --method triple_kg_rag \
  --backend vllm \
  --doc-type dev

[eval] dataset-key: docred_causal
[eval] method: triple_kg_rag
[eval] backend: vllm
[eval] doc-type: dev
[eval] gold: data/preprocessed/docred_causal.jsonl
[eval] preds: /home/galencarmedeiro/git/postdoc/ragtree/data/processed/docred_causal.triple_kg_rag.vllm.jsonl
[eval] ignore-labels: ['null']
[eval] metrics-out: /home/galencarmedeiro/git/postdoc/ragtree/results/relations/docred_causal/triple_kg_rag.vllm.dev.json

=== Micro-level metrics ===
Precision: 0.3291
Recall:    0.1030
F1:        0.1569

=== Counts ===
TP: 1264
FP: 2577
FN: 11011
num_docs_seen: 998
num_docs_eval: 998
num_docs_missing_gold: 0

[eval] Done.


# EventStoryLine

In [ ]:
%run "scripts/build_kg_from_preprocessed.py" \
  --dataset-key eventstoryline \
  --doc-types full \
  --limit 10

#### BYOKG

In [ ]:
%run "scripts/run_kg_rag_relations.py" \
  --dataset-key eventstoryline \
  --backend vllm \
  --doc-type-filter full \
  --kg-path data/kg/eventstoryline__types=full_skip=0_limit=10__kg.json

In [ ]:
%run "scripts/eval_relations.py" \
  --dataset-key eventstoryline \
  --method kg_rag \
  --backend vllm \
  --doc-type full

#### Triple KG RAG

In [12]:
%run "scripts/run_triple_kg_rag_relations.py" \
  --dataset-key eventstoryline \
  --backend vllm \
  --doc-type-filter all \
  --kg-max-hops 1 \
  --kg-max-triples 120 \
  --shot-type all \
  --shot-num 3

[runner] dataset-key=eventstoryline
[runner] method=triple_kg_rag
[runner] backend=vllm, model=openai/gpt-oss-20b
[runner] input=data/preprocessed/eventstoryline.jsonl
[runner] output=/home/galencarmedeiro/git/postdoc/ragtree/data/processed/eventstoryline.triple_kg_rag.vllm.jsonl
[runner] output-format=full
[runner] doc-type-filter=all
[runner] skip=0, limit=None


[triple_kg_rag] Collecting few-shots (type=all): 443doc [00:00, 21865.64doc/s]


[triple_kg_rag] few-shots: requested=3 collected=0 type=all shot_skip=0 shot_limit=None


Running triple_kg_rag on eventstoryline: 443doc [1:18:45, 10.67s/doc]

[runner] Done. Processed 443 documents.
[runner] docs_after_type_filter=443, skip=0, limit=None


In [13]:
%run "scripts/eval_relations.py" \
  --dataset-key eventstoryline \
  --method triple_kg_rag \
  --backend vllm \
  --doc-type all

[eval] dataset-key: eventstoryline
[eval] method: triple_kg_rag
[eval] backend: vllm
[eval] doc-type: all
[eval] gold: data/preprocessed/eventstoryline.jsonl
[eval] preds: /home/galencarmedeiro/git/postdoc/ragtree/data/processed/eventstoryline.triple_kg_rag.vllm.jsonl
[eval] ignore-labels: ['null']
[eval] metrics-out: /home/galencarmedeiro/git/postdoc/ragtree/results/relations/eventstoryline/triple_kg_rag.vllm.all.json

=== Micro-level metrics ===
Precision: 0.2158
Recall:    0.0291
F1:        0.0514

=== Counts ===
TP: 281
FP: 1021
FN: 9359
num_docs_seen: 443
num_docs_eval: 443
num_docs_missing_gold: 0

[eval] Done.


#### CommunityKG

In [21]:
%run "scripts/build_community_kgrag_index.py" \
  --dataset-key eventstoryline \
  --kg-path data/kg/eventstoryline__types=full_skip=0_limit=10__kg.json \
  --output-dir data/kg_community \
  --device cpu

[communitykgrag] Building graph + Louvain communities...
[communitykgrag] Embedding nodes on device=cpu ...


Batches: 100%|██████████| 4/4 [00:00<00:00,  7.48it/s]

[communitykgrag] built at: data/kg_community/eventstoryline
  nodes=119 edges=147 sentences=147 communities=27
  community index: data/kg_community/eventstoryline/faiss.community.index


In [22]:
%run "scripts/run_community_kgrag_relations.py" \
  --dataset-key eventstoryline \
  --communitykg-root data/kg_community \
  --backend vllm \
  --doc-type-filter all \
  --shot-type all \
  --shot-num 3 \
  --top-communities 3 \
  --top-sentences 3 \
  --max-ctx-chars 3000


[runner] dataset-key=eventstoryline
[runner] method=community_kgrag
[runner] backend=vllm, model=openai/gpt-oss-20b
[runner] input=data/preprocessed/eventstoryline.jsonl
[runner] output=/home/galencarmedeiro/git/postdoc/ragtree/data/processed/eventstoryline.community_kgrag.vllm.jsonl
[runner] output-format=full
[runner] doc-type-filter=all
[runner] skip=0, limit=None


[community_kgrag] Collecting few-shots (type=all): 443doc [00:00, 19289.06doc/s]


[community_kgrag] few-shots: requested=3 collected=0 type=all shot_skip=0 shot_limit=None


Running community_kgrag on eventstoryline: 443doc [1:33:38, 12.68s/doc]

[runner] Done. Processed 443 documents.
[runner] docs_after_type_filter=443, skip=0, limit=None


In [23]:
%run "scripts/eval_relations.py" \
  --dataset-key eventstoryline \
  --method community_kgrag \
  --backend vllm \
  --doc-type all

[eval] dataset-key: eventstoryline
[eval] method: community_kgrag
[eval] backend: vllm
[eval] doc-type: all
[eval] gold: data/preprocessed/eventstoryline.jsonl
[eval] preds: /home/galencarmedeiro/git/postdoc/ragtree/data/processed/eventstoryline.community_kgrag.vllm.jsonl
[eval] ignore-labels: ['null']
[eval] metrics-out: /home/galencarmedeiro/git/postdoc/ragtree/results/relations/eventstoryline/community_kgrag.vllm.all.json

=== Micro-level metrics ===
Precision: 0.2640
Recall:    0.0471
F1:        0.0799

=== Counts ===
TP: 454
FP: 1266
FN: 9186
num_docs_seen: 443
num_docs_eval: 443
num_docs_missing_gold: 0

[eval] Done.


# FinCausal

In [ ]:
%run "scripts/build_kg_from_preprocessed.py" \
  --dataset-key fincausal \
  --doc-types train.csv \
  --skip 0

#### BYOKGRAG

In [ ]:
%run "scripts/run_kg_rag_relations.py" \
  --dataset-key fincausal \
  --backend vllm \
  --doc-type-filter all \
  --kg-path data/kg/fincausal__types=train.csv_skip=0_limit=None__kg.json

In [ ]:
%run "scripts/eval_relations.py" \
  --dataset-key fincausal \
  --method kg_rag \
  --backend vllm \
  --doc-type all

#### Triple KG RAG

In [14]:
%run "scripts/run_triple_kg_rag_relations.py" \
  --dataset-key fincausal \
  --backend vllm \
  --doc-type-filter all \
  --kg-max-hops 1 \
  --kg-max-triples 120 \
  --shot-type all \
  --shot-num 3

[runner] dataset-key=fincausal
[runner] method=triple_kg_rag
[runner] backend=vllm, model=openai/gpt-oss-20b
[runner] input=data/preprocessed/fincausal.jsonl
[runner] output=/home/galencarmedeiro/git/postdoc/ragtree/data/processed/fincausal.triple_kg_rag.vllm.jsonl
[runner] output-format=full
[runner] doc-type-filter=all
[runner] skip=0, limit=None


[triple_kg_rag] Collecting few-shots (type=all): 967doc [00:00, 136053.54doc/s]


[triple_kg_rag] few-shots: requested=3 collected=0 type=all shot_skip=0 shot_limit=None


Running triple_kg_rag on fincausal: 967doc [22:53,  1.42s/doc]

[runner] Done. Processed 967 documents.
[runner] docs_after_type_filter=967, skip=0, limit=None


In [18]:
%run "scripts/eval_relations.py" \
  --dataset-key fincausal \
  --method triple_kg_rag \
  --backend vllm \
  --doc-type all

[eval] dataset-key: fincausal
[eval] method: triple_kg_rag
[eval] backend: vllm
[eval] doc-type: all
[eval] gold: data/preprocessed/fincausal.jsonl
[eval] preds: /home/galencarmedeiro/git/postdoc/ragtree/data/processed/fincausal.triple_kg_rag.vllm.jsonl
[eval] ignore-labels: ['null']
[eval] metrics-out: /home/galencarmedeiro/git/postdoc/ragtree/results/relations/fincausal/triple_kg_rag.vllm.all.json

=== Micro-level metrics ===
Precision: 0.9890
Recall:    0.9688
F1:        0.9788

=== Counts ===
TP: 900
FP: 10
FN: 29
num_docs_seen: 967
num_docs_eval: 967
num_docs_missing_gold: 0

[eval] Done.


#### CommunityKG

In [24]:
%run "scripts/build_community_kgrag_index.py" \
  --dataset-key fincausal \
  --kg-path data/kg/fincausal__types=train.csv_skip=0_limit=None__kg.json \
  --output-dir data/kg_community \
  --device cpu

[communitykgrag] Building graph + Louvain communities...
[communitykgrag] Embedding nodes on device=cpu ...


Batches: 100%|██████████| 2/2 [00:00<00:00,  2.37it/s]

[communitykgrag] built at: data/kg_community/fincausal
  nodes=48 edges=24 sentences=24 communities=24
  community index: data/kg_community/fincausal/faiss.community.index


In [25]:
%run "scripts/run_community_kgrag_relations.py" \
  --dataset-key fincausal \
  --communitykg-root data/kg_community \
  --backend vllm \
  --doc-type-filter all \
  --shot-type all \
  --shot-num 3 \
  --top-communities 3 \
  --top-sentences 3 \
  --max-ctx-chars 3000


[runner] dataset-key=fincausal
[runner] method=community_kgrag
[runner] backend=vllm, model=openai/gpt-oss-20b
[runner] input=data/preprocessed/fincausal.jsonl
[runner] output=/home/galencarmedeiro/git/postdoc/ragtree/data/processed/fincausal.community_kgrag.vllm.jsonl
[runner] output-format=full
[runner] doc-type-filter=all
[runner] skip=0, limit=None


[community_kgrag] Collecting few-shots (type=all): 967doc [00:00, 132753.73doc/s]


[community_kgrag] few-shots: requested=3 collected=0 type=all shot_skip=0 shot_limit=None


Running community_kgrag on fincausal: 967doc [25:08,  1.56s/doc]

[runner] Done. Processed 967 documents.
[runner] docs_after_type_filter=967, skip=0, limit=None


In [26]:
# ======== ATTENTION RE-RUN IT =====================================

%run "scripts/eval_relations.py" \
  --dataset-key fincausal \
  --method community_kgrag \
  --backend vllm \
  --doc-type all

[eval] dataset-key: fincausal
[eval] method: community_kgrag
[eval] backend: vllm
[eval] doc-type: all
[eval] gold: data/preprocessed/fincausal.jsonl
[eval] preds: /home/galencarmedeiro/git/postdoc/ragtree/data/processed/fincausal.community_kgrag.vllm.jsonl
[eval] ignore-labels: ['null']
[eval] metrics-out: /home/galencarmedeiro/git/postdoc/ragtree/results/relations/fincausal/community_kgrag.vllm.all.json

=== Micro-level metrics ===
Precision: 0.9923
Recall:    0.9677
F1:        0.9798

=== Counts ===
TP: 899
FP: 7
FN: 30
num_docs_seen: 967
num_docs_eval: 967
num_docs_missing_gold: 0

[eval] Done.


# Maven Ere

In [ ]:
%run "scripts/build_kg_from_preprocessed.py" \
  --dataset-key maven_ere \
  --doc-types train \
  --skip 0

#### BYOKGRAG

In [ ]:
%run "scripts/run_kg_rag_relations.py" \
  --dataset-key maven_ere \
  --backend vllm \
  --doc-type-filter all \
  --kg-path data/kg/maven_ere__types=train_skip=0_limit=None__kg.json

In [ ]:
%run "scripts/eval_relations.py" \
  --dataset-key maven_ere \
  --method kg_rag \
  --backend vllm \
  --doc-type all

#### Triple KG RAG

In [15]:
%run "scripts/run_triple_kg_rag_relations.py" \
  --dataset-key maven_ere \
  --backend vllm \
  --doc-type-filter all \
  --kg-max-hops 1 \
  --kg-max-triples 120 \
  --shot-type all \
  --shot-num 3

[runner] dataset-key=maven_ere
[runner] method=triple_kg_rag
[runner] backend=vllm, model=openai/gpt-oss-20b
[runner] input=data/preprocessed/maven_ere.jsonl
[runner] output=/home/galencarmedeiro/git/postdoc/ragtree/data/processed/maven_ere.triple_kg_rag.vllm.jsonl
[runner] output-format=full
[runner] doc-type-filter=all
[runner] skip=0, limit=None


[triple_kg_rag] Collecting few-shots (type=all): 3516doc [00:00, 8029.90doc/s]


[triple_kg_rag] few-shots: requested=3 collected=0 type=all shot_skip=0 shot_limit=None


Running triple_kg_rag on maven_ere: 3516doc [16:06:49, 16.50s/doc]

[runner] Done. Processed 3516 documents.
[runner] docs_after_type_filter=3516, skip=0, limit=None


In [19]:
%run "scripts/eval_relations.py" \
  --dataset-key maven_ere \
  --method triple_kg_rag \
  --backend vllm \
  --doc-type all

[eval] dataset-key: maven_ere
[eval] method: triple_kg_rag
[eval] backend: vllm
[eval] doc-type: all
[eval] gold: data/preprocessed/maven_ere.jsonl
[eval] preds: /home/galencarmedeiro/git/postdoc/ragtree/data/processed/maven_ere.triple_kg_rag.vllm.jsonl
[eval] ignore-labels: ['null']
[eval] metrics-out: /home/galencarmedeiro/git/postdoc/ragtree/results/relations/maven_ere/triple_kg_rag.vllm.all.json

=== Micro-level metrics ===
Precision: 0.1284
Recall:    0.0352
F1:        0.0553

=== Counts ===
TP: 1621
FP: 11008
FN: 44393
num_docs_seen: 3516
num_docs_eval: 3516
num_docs_missing_gold: 0

[eval] Done.


#### CommunityKGRAG

In [27]:
%run "scripts/build_community_kgrag_index.py" \
  --dataset-key maven_ere \
  --kg-path data/kg/maven_ere__types=train_skip=0_limit=None__kg.json \
  --output-dir data/kg_community \
  --device cpu

[communitykgrag] Building graph + Louvain communities...
[communitykgrag] Embedding nodes on device=cpu ...


Batches: 100%|██████████| 2099/2099 [03:10<00:00, 11.03it/s]


[communitykgrag] built at: data/kg_community/maven_ere
  nodes=67146 edges=36316 sentences=36316 communities=8323
  community index: data/kg_community/maven_ere/faiss.community.index


In [28]:
%run "scripts/run_community_kgrag_relations.py" \
  --dataset-key maven_ere \
  --communitykg-root data/kg_community \
  --backend vllm \
  --doc-type-filter all \
  --shot-type all \
  --shot-num 3 \
  --top-communities 3 \
  --top-sentences 3 \
  --max-ctx-chars 3000


[runner] dataset-key=maven_ere
[runner] method=community_kgrag
[runner] backend=vllm, model=openai/gpt-oss-20b
[runner] input=data/preprocessed/maven_ere.jsonl
[runner] output=/home/galencarmedeiro/git/postdoc/ragtree/data/processed/maven_ere.community_kgrag.vllm.jsonl
[runner] output-format=full
[runner] doc-type-filter=all
[runner] skip=0, limit=None


[community_kgrag] Collecting few-shots (type=all): 3516doc [00:00, 8287.37doc/s]


[community_kgrag] few-shots: requested=3 collected=0 type=all shot_skip=0 shot_limit=None


Running community_kgrag on maven_ere: 3516doc [17:25:19, 17.84s/doc]

[runner] Done. Processed 3516 documents.
[runner] docs_after_type_filter=3516, skip=0, limit=None


In [29]:
%run "scripts/eval_relations.py" \
  --dataset-key maven_ere \
  --method community_kgrag \
  --backend vllm \
  --doc-type all

[eval] dataset-key: maven_ere
[eval] method: community_kgrag
[eval] backend: vllm
[eval] doc-type: all
[eval] gold: data/preprocessed/maven_ere.jsonl
[eval] preds: /home/galencarmedeiro/git/postdoc/ragtree/data/processed/maven_ere.community_kgrag.vllm.jsonl
[eval] ignore-labels: ['null']
[eval] metrics-out: /home/galencarmedeiro/git/postdoc/ragtree/results/relations/maven_ere/community_kgrag.vllm.all.json

=== Micro-level metrics ===
Precision: 0.1315
Recall:    0.0421
F1:        0.0638

=== Counts ===
TP: 1936
FP: 12785
FN: 44078
num_docs_seen: 3516
num_docs_eval: 3516
num_docs_missing_gold: 0

[eval] Done.


# CausalBank

In [ ]:
%run "scripts/build_kg_from_preprocessed.py" \
  --dataset-key causalbank \
  --doc-types resulted_from \
  --skip 0

#### BYOKGRAG

In [ ]:
%run "scripts/run_kg_rag_relations.py" \
  --dataset-key causalbank \
  --backend vllm \
  --doc-type-filter all \
  --kg-path data/kg/causalbank__types=resulted_from_skip=0_limit=None__kg.json

In [ ]:
%run "scripts/eval_relations.py" \
  --dataset-key causalbank \
  --method kg_rag \
  --backend vllm \
  --doc-type all

#### Triple KG RAG

In [16]:
%run "scripts/run_triple_kg_rag_relations.py" \
  --dataset-key causalbank \
  --backend vllm \
  --doc-type-filter all \
  --kg-max-hops 1 \
  --kg-max-triples 120 \
  --shot-type all \
  --shot-num 3

[runner] dataset-key=causalbank
[runner] method=triple_kg_rag
[runner] backend=vllm, model=openai/gpt-oss-20b
[runner] input=data/preprocessed/causalbank.jsonl
[runner] output=/home/galencarmedeiro/git/postdoc/ragtree/data/processed/causalbank.triple_kg_rag.vllm.jsonl
[runner] output-format=full
[runner] doc-type-filter=all
[runner] skip=0, limit=None


[triple_kg_rag] Collecting few-shots (type=all): 1080doc [00:00, 30665.51doc/s]


[triple_kg_rag] few-shots: requested=3 collected=0 type=all shot_skip=0 shot_limit=None


Running triple_kg_rag on causalbank: 1080doc [58:23,  3.24s/doc]

[runner] Done. Processed 1080 documents.
[runner] docs_after_type_filter=1080, skip=0, limit=None


In [17]:
%run "scripts/eval_relations.py" \
  --dataset-key causalbank \
  --method triple_kg_rag \
  --backend vllm \
  --doc-type all

[eval] dataset-key: causalbank
[eval] method: triple_kg_rag
[eval] backend: vllm
[eval] doc-type: all
[eval] gold: data/preprocessed/causalbank.jsonl
[eval] preds: /home/galencarmedeiro/git/postdoc/ragtree/data/processed/causalbank.triple_kg_rag.vllm.jsonl
[eval] ignore-labels: ['null']
[eval] metrics-out: /home/galencarmedeiro/git/postdoc/ragtree/results/relations/causalbank/triple_kg_rag.vllm.all.json

=== Micro-level metrics ===
Precision: 0.7856
Recall:    0.0032
F1:        0.0064

=== Counts ===
TP: 535
FP: 146
FN: 166575
num_docs_seen: 1080
num_docs_eval: 1080
num_docs_missing_gold: 0

[eval] Done.


#### Community KG RAG

In [30]:
%run "scripts/build_community_kgrag_index.py" \
  --dataset-key causalbank \
  --kg-path data/kg/causalbank__types=resulted_from_skip=0_limit=None__kg.json \
  --output-dir data/kg_community \
  --device cpu

[communitykgrag] Building graph + Louvain communities...
[communitykgrag] Embedding nodes on device=cpu ...


Batches: 100%|██████████| 6/6 [00:00<00:00, 10.12it/s]

[communitykgrag] built at: data/kg_community/causalbank
  nodes=173 edges=2402 sentences=2402 communities=10
  community index: data/kg_community/causalbank/faiss.community.index


In [31]:
%run "scripts/run_community_kgrag_relations.py" \
  --dataset-key causalbank \
  --communitykg-root data/kg_community \
  --backend vllm \
  --doc-type-filter all \
  --shot-type all \
  --shot-num 3 \
  --top-communities 3 \
  --top-sentences 3 \
  --max-ctx-chars 3000


[runner] dataset-key=causalbank
[runner] method=community_kgrag
[runner] backend=vllm, model=openai/gpt-oss-20b
[runner] input=data/preprocessed/causalbank.jsonl
[runner] output=/home/galencarmedeiro/git/postdoc/ragtree/data/processed/causalbank.community_kgrag.vllm.jsonl
[runner] output-format=full
[runner] doc-type-filter=all
[runner] skip=0, limit=None


[community_kgrag] Collecting few-shots (type=all): 1080doc [00:00, 27899.34doc/s]


[community_kgrag] few-shots: requested=3 collected=0 type=all shot_skip=0 shot_limit=None


Running community_kgrag on causalbank: 1080doc [1:30:08,  5.01s/doc]

[runner] Done. Processed 1080 documents.
[runner] docs_after_type_filter=1080, skip=0, limit=None


In [32]:
%run "scripts/eval_relations.py" \
  --dataset-key causalbank \
  --method community_kgrag \
  --backend vllm \
  --doc-type all

[eval] dataset-key: causalbank
[eval] method: community_kgrag
[eval] backend: vllm
[eval] doc-type: all
[eval] gold: data/preprocessed/causalbank.jsonl
[eval] preds: /home/galencarmedeiro/git/postdoc/ragtree/data/processed/causalbank.community_kgrag.vllm.jsonl
[eval] ignore-labels: ['null']
[eval] metrics-out: /home/galencarmedeiro/git/postdoc/ragtree/results/relations/causalbank/community_kgrag.vllm.all.json

=== Micro-level metrics ===
Precision: 0.8011
Recall:    0.0044
F1:        0.0087

=== Counts ===
TP: 733
FP: 182
FN: 166377
num_docs_seen: 1080
num_docs_eval: 1080
num_docs_missing_gold: 0

[eval] Done.
